# HRAI API demo notebook

In [ ]:
!pip install --upgrade pip
!pip install -r requirements.txt

!git lfs pull

In [1]:
import os, json, random
from pathlib import Path
import requests
from rich import print as rprint
from dotenv import load_dotenv

from pos_extraction import text_to_ngrams
from models.RequestModel import ManualContentRequest, QueryRequest


BASE_URL = 'http://127.0.0.1:8000'

In [2]:
# pro rychlejší fungování encoderu doporučuji nastavit HF_TOKEN v .env:
load_dotenv()

True

## převedení textu na ngramy

In [3]:
resumes = json.loads(Path(os.path.join('testfiles', 'cs_resumes.json')).read_text(encoding='utf-8'))
text = random.choice(resumes)
print(text)

Kreativní a inovativní kreslič, vášeň pro navrhování budov a pokročilé konstrukční řešení. |Hledající nejlepší příležitost k dalšímu prohloubení stávajících profesních zkušeností a rozšiřovat znalostní základnu o architektonických návrzích a zároveň růst s organizace. Dovednosti 3D modelování, architektonické kreslení, AutoCAD, čtení plánů, tesařství, Orientovaný na detaily, kreslení, e-mail, půdorysy, rám, 3dsMax, mechanické, zasílání zpráv, aplikace Microsoft Office, Multi-tasking, malování, dovednosti při řešení problémů, čtení, renovace, Revit, sebemotivovaný, supervizor Zkušenosti Inženýrský technik Červenec až červen Název společnosti - Město, Stát vytváření a dokončování výkresů na stavebních plánech. Projděte si projekty s konstruktéry a konstruktéry těsnění a najděte řešení problémů. Pomáháme zákazníkům v terénu prostřednictvím telefonátů do prodejen, e-mailů nebo instant messagingu. Multitasking mezi různými úlohami denně. Material Handler Červenec až prosinec Název společnos

In [4]:
ngrams = text_to_ngrams(text)
print(', '.join(sorted(ngrams)))

3 d modelování, 3 dsmax, 3.4 perfektní docházka, aas, aas kreslení, akademické vyznamenání, akademické vyznamenání 3.4, aplikace, aplikace microsoft, aplikace microsoft office, architektonické kreslení, architektonické kreslení autocad, architektonických návrzích, areálu, autocad, autocad čtení, autocad čtení plánů, automobilů, balíků, bim udržitelnost, budov, budovy, budovy včetně, budovy včetně malování, cedule, cedule kolem, cedule kolem kampusu, d modelování, dalšímu prohloubení, designu další informace, detaily, detaily kreslení, detaily kreslení e, docházka, dokončování, dokončování výkresů, dovednosti, dovednosti 3, dovednosti 3 d, dovednosti při řešení, dsmax, dsmax mechanické zasílání, dělník, dělník červen, e mail, e mail půdorysy, e mailů, hledající nejlepší příležitost, informace, inovativní kreslič, inovativní kreslič vášeň, instalováno cedule, instalováno cedule kolem, instant, instant messagingu, institut, institut itt, institut itt město, inženýrský technik, inženýrský 

## Informace z životopisu se zaměřením
`POST /resume` — nahraje soubor + volitelný `target_job`
→ `SuggestionResponse` (`top_suggestion`, `target_suggestion`)

In [35]:
resume_path = os.path.join('testfiles', 'Životopis7.pdf')
target_job = 'rybář'

with open(resume_path, 'rb') as f:
    files = {'file': (os.path.basename(resume_path), f)}
    data = {'target_job': target_job}

    req = requests.post(f'{BASE_URL}/resume', files=files, data=data)

r = req.json()

In [36]:
if r['top_suggestion']:
    missing = set([s['label'] for s in r['top_suggestion']['missing_skills']])
    goal = r['top_suggestion']['occupation']
    lbl_ = goal['label']
    print(
        f'skore zadané pozice z dovedností: {round(goal['score'],5)} ({lbl_})\n\n'
        f'další dovednosti k {lbl_}:\n{'\n'.join(missing)}'
    )


In [37]:
if r['target_suggestion']:
    missing_ = set([s['label'] for s in r['target_suggestion']['missing_skills']])
    goal = r['target_suggestion']['occupation']
    lbl_ = goal['label']
    print(
        f'skore zadané pozice z dovedností: {round(goal['score'],5)} ({lbl_})\n\n'
        f'další dovednosti k {lbl_}:\n{'\n'.join(missing_)}'
    )

skore zadané pozice z dovedností: 0.68068 (kapitán rybářské lodi/kapitánka rybářské lodi)

další dovednosti k kapitán rybářské lodi/kapitánka rybářské lodi:
používat rybářské manévry
vést a uchovávat deníky
rozumět plánům ukládání
zvládat náročné situace v odvětví rybolovu
globální námořní tísňový a bezpečnostní systém
chovat se k cestujícím přátelsky
koordinovat nakládání s rybami
posuzovat stabilitu plavidel
obsluhovat záchranné vybavení lodi
informovat cestující
přežít na moři v případě opuštění lodi
zajistit pohodlí cestujících
sledovat úroveň zásob
koordinovat cestující
používat námořní angličtinu
vytvářet časový plán rybolovu
průběžně rozvíjet své profesní znalosti v oblasti rybolovu
zajistit komunikaci v terénu
všeobecné programy a kvalifikace
rybářská plavidla
vytvářet plány ukládání
zajistit uložený náklad
přijmout opatření pro bezpečnou plavbu
poskytnout první pomoc
vést plavbu
vyhodnocovat hejna ryb
používat různé komunikační kanály
zajistit shodu plavidla s předpisy
hasit p

## 2. Analýza životopisu → práce z různých oblastí pracovního trhu
`POST /resume/domains` — nahraje soubor
→ `DomainResponse` (`domains`: list of `DomainResult`, each with `domain`, `occ_count`, `occupations`)

In [59]:
resume_path = os.path.join('testfiles', 'resume.docx')

with open(resume_path, 'rb') as f:
    files = {'file': (os.path.basename(resume_path), f)}
    r = requests.post(f'{BASE_URL}/resume/domains', files=files)


In [60]:
rprint(r.json()) #TODO fix

{'domains': []}

## 3. Dovednosti → povolání dle skóre
`POST /text` — JSON s klíčem `skills` (čárkou oddělené dovednosti) + volitelný `target_job`
→ `SkillsResponse` (`suggestions`: list of `Suggestion`)

In [40]:
payload = ManualContentRequest(
    skills='Python, machine learning, analýza dat, SQL, statistika',
    target_job='Business analytik'
)
r = requests.post(f'{BASE_URL}/text', json=payload.model_dump())
print(f'Status: {r.status_code}')

Status: 200


In [ ]:
rprint(r.json())

## 4. Skóre zadaného povolání
`POST /text/goal` — JSON s `skills` + `target_job`
→ `TargetJobResponse` (`top_suggestion`, `target_suggestion`)

In [42]:
payload = ManualContentRequest(
    skills='Python, machine learning, analýza dat, SQL, statistika, matematické modelování, práce s databázemi',
    target_job='datový analytik'
)
r = requests.post(f'{BASE_URL}/text/goal', json=payload.model_dump())

print(f'Status: {r.status_code}')

Status: 200


In [ ]:
rprint(r.json())

## 5. Vyhledávání individuálních entit
`POST /query` — JSON s `text`, volitelný `entity_type` (`all`/`occupation`/`skill`/`skill_group`/`isco_group`), volitelný `n`
→ `List[EntityResult]`

In [57]:
query = QueryRequest(
    text='asistent v právnické kanceláři',
    entity_type='occupation',
    n=5
)
r = requests.post(f'{BASE_URL}/query', json=query.model_dump())

print(f'Status: {r.status_code}')

Status: 200


In [58]:
rprint(r.json())

[
    {
        'id': 6360,
        'cosine_score': 0.8662424087524414,
        'entity_type': 'occupation',
        'esco_uri': 'http://data.europa.eu/esco/occupation/929ef41d-afd0-4d5a-b85b-69eab68fa368',
        'label': 'právní asistent/právní asistentka',
        'code': '3411.7',
        'isco_code': 3411.0,
        'description': 'Právní asistenti úzce spolupracují s právníky a právními zástupci při právním výzkumu a při
přípravě soudních řízení. Pomáhají s dokumentací případů a se správou administrativní stránky soudních 
záležitostí.'
    },
    {
        'id': 6361,
        'cosine_score': 0.8589240312576294,
        'entity_type': 'occupation',
        'esco_uri': 'http://data.europa.eu/esco/occupation/929ef41d-afd0-4d5a-b85b-69eab68fa368',
        'label': 'právní asistent/právní asistentka',
        'code': '3411.7',
        'isco_code': 3411.0,
        'description': 'Právní asistenti úzce spolupracují s právníky a právními zástupci při právním výzkumu a při
přípravě soudních řízení. Pomáhají s dokumentací případů a se správou administrativní stránky soudních 
záležitostí.'
    },
    {
        'id': 122,
        'cosine_score': 0.8287259936332703,
        'entity_type': 'occupation',
        'esco_uri': 'http://data.europa.eu/esco/occupation/0264b794-3771-47aa-a675-78bf13b62ee7',
        'label': 'asistent v právní administrativě/asistentka v právní administrativě',
        'code': '3342.2',
        'isco_code': 3342.0,
        'description': 'Asistenti v právní administrativě vykonávají každodenní administrativní činnost pro firmy, 
veřejné kanceláře a podniky. Vykonávají činnosti, jako je pořizování zápisu, psaní mailů a odpovídání na telefonní 
hovory. Při těchto činnostech využívají specifické znalosti postupů a předpisů v obchodní právní administrativě.'
    },
    {
        'id': 121,
        'cosine_score': 0.8270215392112732,
        'entity_type': 'occupation',
        'esco_uri': 'http://data.europa.eu/esco/occupation/0264b794-3771-47aa-a675-78bf13b62ee7',
        'label': 'asistent v právní administrativě/asistentka v právní administrativě',
        'code': '3342.2',
        'isco_code': 3342.0,
        'description': 'Asistenti v právní administrativě vykonávají každodenní administrativní činnost pro firmy, 
veřejné kanceláře a podniky. Vykonávají činnosti, jako je pořizování zápisu, psaní mailů a odpovídání na telefonní 
hovory. Při těchto činnostech využívají specifické znalosti postupů a předpisů v obchodní právní administrativě.'
    },
    {
        'id': 6647,
        'cosine_score': 0.7901760935783386,
        'entity_type': 'occupation',
        'esco_uri': 'http://data.europa.eu/esco/occupation/974d4eab-fbee-4c52-b16e-73bdd8c25b53',
        'label': 'právník/právnička',
        'code': '2611.1',
        'isco_code': 2611.0,
        'description': 'Právníci poskytují právní poradenství klientům a jednají jejich jménem při soudních 
řízeních a v souladu s právními předpisy. Zkoumají, vykládají a studují případy, aby mohli zastupovat své klienty v
různých prostředích, jako jsou soudy a správní rady. Předkládají jménem svých klientů argumenty k soudním řízením v
různých souvislostech s cílem nalézt opravné prostředky.'
    }
]

### Vyhledávání dovedností

In [3]:
query = QueryRequest(
    text='vymýšlení absurdních testovacích dat',
    entity_type='skill',
    n=5
)
r = requests.post(f'{BASE_URL}/query', json=query.model_dump())

print(f'Status: {r.status_code}')

Status: 200


In [4]:
rprint(r.json())

[
    {
        'id': 8395,
        'cosine_score': 0.6693181991577148,
        'entity_type': 'skill',
        'esco_uri': 'http://data.europa.eu/esco/skill/81a2db2c-7e55-44d0-9cd9-74c25147d7cd',
        'label': 'analyzovat údaje ze zkoušek',
        'code': '',
        'isco_code': '',
        'description': 'Interpretovat a analyzovat údaje shromážděné během zkoušek za účelem vyvození závěrů, 
nových poznatků nebo řešení.'
    }
]

### Vyhledávání ve všech entitách
občas to zvládne i dost niche věci

In [5]:
query = QueryRequest(
    text='hodnocení ročníkových prací studentů',
    entity_type='all',
    n=20
)
r = requests.post(f'{BASE_URL}/query', json=query.model_dump())

print(f'Status: {r.status_code}')

Status: 200


In [6]:
rprint(r.json())


[
    {
        'id': 18110,
        'cosine_score': 0.7254202365875244,
        'entity_type': 'skill',
        'esco_uri': 'http://data.europa.eu/esco/skill/6fe019dd-027f-45f3-b19c-d27f7ae00980',
        'label': 'hodnotit studenty',
        'code': '',
        'isco_code': '',
        'description': 'Hodnotit (akademický) pokrok, dosažené výsledky, znalosti a dovednosti studentů 
prostřednictvím úkolů, zkoušek a testů. Rozpoznat jejich potřeby a sledovat jejich pokrok a silné a slabé stránky. 
Formulovat souhrnné prohlášení o cílech, kterých student dosáhl.'
    }
]